In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import torch
from hydra.utils import instantiate
from hydra import initialize, compose
import hydra
import wandb

from data.dataManager import DataManager
from model.modelCreator import ModelCreator
from omegaconf import OmegaConf
from scripts.run import setup_model, load_model_instance

In [ ]:
hydra.core.global_hydra.GlobalHydra.instance().clear()
initialize(version_base=None, config_path="config")
config=compose(config_name="config_layers.yaml")
wandb.init(tags = [config.data.dataset_name], project=config.wandb.project, entity=config.wandb.entity, config=OmegaConf.to_container(config, resolve=True), mode='disabled')

In [ ]:
devids = ["cuda:{0}".format(x) for x in list(config.gpu_list)]
dev = torch.device(devids[0])

In [ ]:
new_model = True
if new_model:
    self = setup_model(config)
    # self.model = self.model.double()  # sets all model parameters to float64
else:
    self = load_model_instance(config.config_path)
    # self.model = self.model.double()

In [ ]:
# collect u vectors from dataloader, concat and plot histograms for each of u_i
u_list = []
for (x, x0, u, E) in self.data_mgr.val_loader:
    u_list.append(u.cpu().numpy())

u = np.concatenate(u_list, axis=0)
print(u.shape)
print(type(u_list[0][0][0]))
plt.figure(figsize=(12, 6))
for i in range(5):
    plt.subplot(2, 3, i+1)
    plt.hist(u[:, i], bins=30, alpha=0.7, color='blue', edgecolor='black')
    plt.title(rf'$U_{i+1}$ Distribution')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.grid(axis='y', linestyle='--', alpha=0.4)
    plt.xlim(0, 1)
    plt.yscale('log')
plt.tight_layout()
print(u.min(), u.max())


In [ ]:
#print(self.model.encoder.gray_codec.tables.keys())
print(hasattr(self._config, 'use_gray_code') and self._config.use_gray_code)

In [ ]:
self.evaluate_ae(self.data_mgr.val_loader, epoch=0)

In [ ]:
# Force a forward pass (or call the encoding function directly)
# This triggers _get_or_create_table for 19, 17, and 17 bits
print(self.incident_energy.shape)
self.model.encoder.gray_energy_encoding(self.incident_energy)

# NOW check the keys
print(self.model.encoder.gray_codec.tables.keys())
# Output should be: dict_keys(['19', '17'])
table_19 = self.model.encoder.gray_codec.tables['19']

# If it's Standard Gray, the first column (MSB) will be:
# 0 0 0 ... 1 1 1
print(table_19[:, 0])
# Calculate absolute difference between adjacent rows
flips = (table_19[:-1, 0] != table_19[1:, 0]).sum().item()

print(f"Number of flips: {flips}")
midpoint = 2**19 // 2 

# Slice 10 rows before and 10 rows after the midpoint
print("Middle values of MSB:")
print(table_19[midpoint-10 : midpoint+10, 0])
# Calculate flips for EVERY column in the 19-bit table
table_19 = self.model.encoder.gray_codec.tables['19']
all_flips = (table_19[:-1] != table_19[1:]).sum(dim=0)

print("Flip counts per column (Should be random, NOT sorted):")
print(all_flips)

# Check if the "1 flip" column (the true MSB) exists somewhere
print(f"\nLocation of the true MSB (1 flip): Index {(all_flips == 1).nonzero().item()}")

In [ ]:
print(self.generate_plots(0, "ae"))

In [ ]:
from model.gumbel import GumbelNoNoise
from torch import nn
class LatentIdentity(nn.Module):
    """
    The ultimate sanity check. No sigmoid, no binary constraints.
    Passes raw, unbounded continuous logits straight into the decoder.
    """
    def __init__(self):
        super().__init__()
        
    def forward(self, logits, beta=None):
        return logits
total_epochs = 20000
self.optimiser.param_groups[0]['lr'] = 1e-3
self._config.engine.beta_gumbel_duration = total_epochs
# self.model.encoder.smoothing_dist_mod = GumbelNoNoise()

for epoch in range(total_epochs):
    self.fit_ae(epoch)

In [ ]:
self.evaluate_ae(self.data_mgr.train_loader, epoch=0)

In [ ]:
print(self.post_samples.mean(dim=0))

In [ ]:
self.generate_plots(0, close_plots=False)

In [ ]:
print(self.model.decoder.n_latent_nodes)

In [ ]:
print("total number of parameters:", np.log(sum(p.numel() for p in self.model.parameters() if p.requires_grad))) #e^19 for large model

In [ ]:
from utils.optimization.orchestrator import EvaluationOrchestrator

orchestrator = EvaluationOrchestrator(
    base_cfg=self._config,
    model=self.model,
    reduce_fn=self._reduce,
    inv_reduce_fn=self._reduceinv,
    device=self.device
)

objective_value = orchestrator.evaluate_objective()
print("Objective Value:", objective_value)